# 規約の新旧を条項ごとに比べ、改定の要点をまとめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_summarization/contract_diff.ipynb)

NLP クックブック 24-09 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

利用規約や契約書の新旧 2 つの版を比べて、追加・削除・変更された条項を一覧にし、変更の要点を短くまとめたい。条番号がずれても正しく対応づけ、「できる」が「しなければならない」に変わるような小さな変更も見落とさないようにしたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import re
import unicodedata
import pandas as pd

# 説明のために作った架空の規約 (実在のサービスの規約ではない)。新版は条番号を全角で書いてある
OLD = """サンプル会員サービス利用規約 (2025 年 4 月 1 日版)
第1条（適用）
本規約は、当社が提供するサンプル会員サービス（以下「本サービス」といいます。）の利用に関する条件を定めるものです。
第2条（会員登録）
本サービスの利用を希望する者は、当社の定める方法により会員登録を申請することができます。登録は、当社が申請を承認した時点で完了します。
第3条（料金及び支払方法）
会員は、本サービスの利用料金として月額500円（税込）を、当社が指定する方法により支払うものとします。
第4条（禁止事項）
会員は、本サービスの利用にあたり、次の行為をしてはなりません。
(1) 法令に違反する行為
(2) 他の会員の情報を収集する行為
(3) 本サービスの運営を妨げる行為
第5条（パスワードの管理）
会員は、自己の責任においてパスワードを管理するものとします。パスワードが第三者に知られたおそれがある場合、会員は当社に通知することができます。
第6条（本サービスの変更）
当社は、会員に事前に通知することにより、本サービスの内容を変更することができます。
第7条（退会）
会員は、当社所定の方法により届け出ることにより、いつでも退会することができます。会員が第4条に違反した場合、当社はその会員を退会させることができます。
第8条（ポイント）
当社は、本サービスの利用に応じて会員にポイントを付与します。ポイントの有効期限は、付与日から1年間とします。
第9条（免責）
当社は、本サービスに関して会員に生じた損害について、当社に故意又は重大な過失がある場合を除き、責任を負いません。
第10条（準拠法及び管轄）
本規約には日本法を適用し、本サービスに関する紛争については、東京地方裁判所を第一審の専属的合意管轄裁判所とします。
"""

NEW = """サンプル会員サービス利用規約 (2026 年 4 月 1 日版)
第１条（適用）
本規約は、当社が提供するサンプル会員サービス（以下「本サービス」といいます。）の利用に関する条件を定めるものです。
第２条（定義）
本規約において「会員」とは、第３条に基づき会員登録が完了した者をいい、「投稿データ」とは、会員が本サービスに投稿した文章、画像その他の情報をいいます。
第３条（会員登録）
本サービスの利用を希望する者は、当社の定める方法により会員登録を申請することができます。登録は、当社が申請を承認した時点で完了します。
第４条（料金及び支払方法）
会員は、本サービスの利用料金として月額550円（税込）を、当社が指定する方法により支払うものとします。
第５条（禁止事項）
会員は、本サービスの利用にあたり、次の行為をしてはなりません。
(1) 法令に違反する行為
(2) 他の会員の情報を収集する行為
(3) 本サービスの運営を妨げる行為
(4) 本サービスで提供される情報を、当社の許可なく機械学習に利用する行為
第６条（パスワードの管理）
会員は、自己の責任においてパスワードを管理するものとします。パスワードが第三者に知られたおそれがある場合、会員は当社に通知しなければなりません。
第７条（本サービスの変更）
当社は、会員に事前に通知することにより、本サービスの内容を変更することができます。
第８条（退会）
会員は、当社所定の方法により届け出ることにより、いつでも退会することができます。会員が第５条に違反した場合、当社はその会員を退会させることができます。
第９条（免責事項）
当社は、本サービスに関して会員に生じた損害について、当社に故意がある場合を除き、責任を負いません。
第１０条（準拠法及び管轄）
本規約には日本法を適用し、本サービスに関する紛争については、東京地方裁判所を第一審の専属的合意管轄裁判所とします。
"""

def parse_articles(doc):
    """「第N条（見出し）」の行で区切り、条番号・見出し・本文の一覧にする。全角の数字や括弧は NFKC で半角にそろえる"""
    doc = unicodedata.normalize("NFKC", doc)
    parts = re.split(r"^第(\d+)条\s*\((.+?)\)\s*$", doc, flags=re.M)
    return [{"no": int(n), "title": t, "body": "\n".join(l.strip() for l in b.strip().splitlines())}
            for n, t, b in zip(parts[1::3], parts[2::3], parts[3::3])]

old, new = parse_articles(OLD), parse_articles(NEW)
print(len(old), "条 →", len(new), "条")

# 素朴な方法: 同じ条番号どうしを比べる
old_by_no, new_by_no = {a["no"]: a for a in old}, {a["no"]: a for a in new}
pd.DataFrame([{"条": n, "旧": old_by_no.get(n, {}).get("title"), "新": new_by_no.get(n, {}).get("title"),
               "本文が同じか": old_by_no.get(n, {}).get("body") == new_by_no.get(n, {}).get("body")} for n in sorted(old_by_no | new_by_no)])

10 条 → 10 条


,条,旧,新,本文が同じか
0,1,適用,適用,True
1,2,会員登録,定義,False
2,3,料金及び支払方法,会員登録,False
3,4,禁止事項,料金及び支払方法,False
4,5,パスワードの管理,禁止事項,False
5,6,本サービスの変更,パスワードの管理,False
6,7,退会,本サービスの変更,False
7,8,ポイント,退会,False
8,9,免責,免責事項,False
9,10,準拠法及び管轄,準拠法及び管轄,True


In [3]:
import difflib
import numpy as np
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("intfloat/multilingual-e5-small")   # MIT ライセンス

def mask_refs(s):
    return re.sub(r"第\d+条", "第N条", s)        # 対応づけの段階では、条番号への参照を伏せる

def view(a):
    return f"{a['title']}\n{mask_refs(a['body'])}"

E_old = emb_model.encode(["query: " + view(a) for a in old], normalize_embeddings=True)
E_new = emb_model.encode(["query: " + view(a) for a in new], normalize_embeddings=True)
emb_sim = E_old @ E_new.T                       # 意味の近さ (言い換えや見出しの変更に強い)
char_sim = np.array([[difflib.SequenceMatcher(None, view(a), view(b), autojunk=False).ratio() for b in new] for a in old])   # 文字列の重なり
score = (emb_sim + char_sim) / 2

# 割当問題として解き、スコアの合計が最大になる 1 対 1 の組を選ぶ (ハンガリアン法などで解ける問題)
rows, cols = linear_sum_assignment(-score)
THRESHOLD = 0.75                                # これより低い組は対応なし (削除と新設) とみなす
pairs = [(i, j) for i, j in zip(rows, cols) if score[i, j] >= THRESHOLD]
# 対応した組と、それ以外の組の類似度の差を、3 つの尺度で比べる
is_pair = np.zeros_like(score, dtype=bool)
for i, j in pairs:
    is_pair[i, j] = True
for label, m in [("文埋め込み", emb_sim), ("文字列", char_sim), ("平均", score)]:
    print(f"{label}: 対応した組の最小 {m[is_pair].min():.2f} / それ以外の最大 {m[~is_pair].max():.2f}")
pd.DataFrame(score.round(2), index=[f"旧{a['no']} {a['title']}" for a in old], columns=[f"新{a['no']}" for a in new])

文埋め込み: 対応した組の最小 0.98 / それ以外の最大 0.94
文字列: 対応した組の最小 0.82 / それ以外の最大 0.41
平均: 対応した組の最小 0.90 / それ以外の最大 0.66


,新1,新2,新3,新4,新5,新6,新7,新8,新9,新10
旧1 適用,1.00,0.59,0.54,0.56,0.56,0.54,0.55,0.57,0.57,0.60
旧2 会員登録,0.60,0.55,1.00,0.65,0.55,0.53,0.64,0.66,0.57,0.55
旧3 料金及び支払方法,0.60,0.55,0.65,0.99,0.58,0.56,0.61,0.58,0.61,0.58
旧4 禁止事項,0.56,0.56,0.57,0.59,0.90,0.55,0.56,0.57,0.59,0.53
旧5 パスワードの管理,0.53,0.51,0.58,0.57,0.54,0.95,0.62,0.57,0.57,0.49
旧6 本サービスの変更,0.55,0.53,0.65,0.61,0.54,0.55,1.00,0.62,0.60,0.56
旧7 退会,0.57,0.57,0.66,0.58,0.54,0.61,0.62,1.00,0.56,0.50
旧8 ポイント,0.58,0.53,0.59,0.60,0.54,0.52,0.59,0.52,0.62,0.57
旧9 免責,0.57,0.56,0.57,0.59,0.56,0.58,0.60,0.53,0.95,0.58
旧10 準拠法及び管轄,0.60,0.56,0.56,0.59,0.51,0.50,0.56,0.50,0.59,1.00


In [4]:
from collections import Counter
from fugashi import Tagger

tagger = Tagger()
o2n = {old[i]["no"]: new[j]["no"] for i, j in pairs}

def remap_refs(s):
    """旧版の「第N条」を、対応する新版の条番号に読み替える (番号の繰り下げだけによる差分を消す)"""
    return re.sub(r"第(\d+)条", lambda m: f"第{o2n.get(int(m.group(1)), m.group(1))}条", s)

def word_diff(a, b):
    """形態素の単位で差分を取り、削除を [-…-]、追加を {+…+} で示す"""
    wa, wb = [w.white_space + w.surface for w in tagger(a)], [w.white_space + w.surface for w in tagger(b)]
    out = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, wa, wb, autojunk=False).get_opcodes():
        if op == "equal":
            out.append("".join(wa[i1:i2]))
        else:
            out.append(("[-" + "".join(wa[i1:i2]) + "-]") * (i2 > i1) + ("{+" + "".join(wb[j1:j2]) + "+}") * (j2 > j1))
    return "".join(out)

# 権利や義務の強さを決める語句と、数値。文全体の類似度ではほとんど変わらないので、別に数えて比べる
KEY_PATTERNS = {"義務": r"しなければな(?:らない|りません)", "禁止": r"してはな(?:らない|りません)", "できる(権利・任意)": r"ことができ(?:る|ます)",
                "努力義務": r"努め(?:る|ます)", "免責": r"責任を負(?:わない|いません)", "故意": r"故意", "重大な過失": r"重大な過失",
                "事前に": r"事前に", "直ちに": r"直ちに", "いつでも": r"いつでも"}

def key_terms(s):
    c = Counter({k: len(re.findall(p, s)) for k, p in KEY_PATTERNS.items()})
    c.update(re.findall(r"\d+(?:\.\d+)?(?:円|%|日|か月|年間|年|時間|件)", mask_refs(s)))
    return +c

def term_changes(a, b):
    ca, cb = key_terms(a), key_terms(b)
    return ", ".join([f"-{k}" for k in (ca - cb)] + [f"+{k}" for k in (cb - ca)])

def row(old_no, new_no, status, title, old_text, new_text, terms="", diff=""):
    return {"旧": old_no, "新": new_no, "区分": status, "見出し": title, "要注意の語句": terms, "差分": diff,
            "old_text": old_text, "new_text": new_text}

report = []
for i, j in sorted(pairs, key=lambda p: p[1]):
    a, b = old[i], new[j]
    a_body = remap_refs(a["body"])
    title = b["title"] if a["title"] == b["title"] else f"{a['title']}→{b['title']}"
    if (a["title"], a_body) == (b["title"], b["body"]):
        report.append(row(a["no"], b["no"], "変更なし", title, a_body, b["body"]))
    else:
        report.append(row(a["no"], b["no"], "変更", title, a_body, b["body"],
                          term_changes(a_body, b["body"]), word_diff(a_body, b["body"])))
matched_old, matched_new = {i for i, _ in pairs}, {j for _, j in pairs}
report += [row(None, b["no"], "新設", b["title"], "", b["body"]) for j, b in enumerate(new) if j not in matched_new]
report += [row(a["no"], None, "削除", a["title"], a["body"], "") for i, a in enumerate(old) if i not in matched_old]
report = pd.DataFrame(report).astype({"旧": "Int64", "新": "Int64"})

# 変更のあった条の差分は、切れないように全文を表示する
for _, r in report[report["区分"] == "変更"].iterrows():
    print(f"旧{r['旧']}→新{r['新']} {r['見出し']}\n{r['差分']}\n")
report.drop(columns=["差分", "old_text", "new_text"])

旧3→新4 料金及び支払方法
会員は、本サービスの利用料金として月額[-500-]{+550+}円(税込)を、当社が指定する方法により支払うものとします。

旧4→新5 禁止事項
会員は、本サービスの利用にあたり、次の行為をしてはなりません。
(1) 法令に違反する行為
(2) 他の会員の情報を収集する行為
(3) 本サービスの運営を妨げる行為{+
(4) 本サービスで提供される情報を、当社の許可なく機械学習に利用する行為+}

旧5→新6 パスワードの管理
会員は、自己の責任においてパスワードを管理するものとします。パスワードが第三者に知られたおそれがある場合、会員は当社に通知[-することができます-]{+しなければなりません+}。

旧9→新9 免責→免責事項
当社は、本サービスに関して会員に生じた損害について、当社に故意[-又は重大な過失-]がある場合を除き、責任を負いません。



,旧,新,区分,見出し,要注意の語句
0,1,1,変更なし,適用,
1,2,3,変更なし,会員登録,
2,3,4,変更,料金及び支払方法,"-500円, +550円"
3,4,5,変更,禁止事項,
4,5,6,変更,パスワードの管理,"-できる(権利・任意), +義務"
5,6,7,変更なし,本サービスの変更,
6,7,8,変更なし,退会,
7,9,9,変更,免責→免責事項,-重大な過失
8,10,10,変更なし,準拠法及び管轄,
9,<NA>,2,新設,定義,


In [5]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # SB Intuitions、MIT ライセンス。初回はダウンロード (約 6.4GB) を含めて数分。VRAM 約 7GB
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(prompt, max_new_tokens=150):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def summarize_change(r):
    if r["区分"] == "新設":
        return chat(f"利用規約の改定で、次の条項が新しく設けられました。何が定められたかを 1 文で書いてください。\n\n（{r['見出し']}）{r['new_text']}")
    if r["区分"] == "削除":
        return chat(f"利用規約の改定で、次の条項が削除されました。何がなくなったかを 1 文で書いてください。\n\n（{r['見出し']}）{r['old_text']}")
    return chat("利用規約の条項が改定されました。旧版と新版、差分 ([-削除-] {+追加+})、機械的に検出した語句の変化を示します。"
                "何がどう変わったかを 1 文で、会員にとって有利か不利かを 1 文で書いてください。差分に表れた変更は、小さなものでも省かないでください。\n\n"
                f"旧版: {r['old_text']}\n新版: {r['new_text']}\n差分: {r['差分']}\n語句の変化: {r['要注意の語句'] or 'なし'}")

changed = report[report["区分"] != "変更なし"].copy()
changed["要点"] = [summarize_change(r) for _, r in changed.iterrows()]   # 6 条の生成はモデルの読み込みと合わせて A5000 で約 10 秒 (キャッシュ済み)。T4 ではこれより遅い

def no(x):
    return "-" if pd.isna(x) else x

for _, r in changed.iterrows():
    print(f"[{r['区分']}] 旧{no(r['旧'])}→新{no(r['新'])} {r['見出し']}  {r['要注意の語句']}")
    print(f"   {r['要点']}")

[変更] 旧3→新4 料金及び支払方法  -500円, +550円
   利用規約において、会員は本サービスの利用料金として月額500円から550円（税込）に変更され、会員にとっては不利になった。具体的には、月額料金が50円増額された。
[変更] 旧4→新5 禁止事項  
   利用規約の条項において、会員は本サービスの利用にあたり法令違反や他の会員の情報収集、サービス運営妨害に加えて、当社が許可しない形で本サービスで提供される情報を機械学習に利用することも禁止されるようになりました。この変更は会員にとって不利であり、特にデータの利用に関する制限が強化された形となります。
[変更] 旧5→新6 パスワードの管理  -できる(権利・任意), +義務
   利用規約の条項において、会員はパスワードを自己管理する責任があり、パスワードが第三者に知られたおそれがある場合に通知する義務が追加された。これにより、会員にとっては通知義務が生じるため、不利に変更されたと言える。
[変更] 旧9→新9 免責→免責事項  -重大な過失
   旧版では、当社が本サービスに関して会員に生じた損害に対して重大な過失がない限り責任を負わないとされていましたが、新版では故意がない限り責任を負わないという変更がなされました。この改定により、会員にとっては責任の範囲が狭まったため、不利になる可能性があります。
[新設] 旧-→新2 定義  
   新しく定められた条項は、会員と投稿データの定義に関するもので、会員を会員登録が完了した者とし、投稿データを会員が本サービスに投稿した文章、画像その他の情報として明確に定義しています。
[削除] 旧8→新- ポイント  
   利用規約から削除されたのは、ポイント制度に関する条項で、具体的には「当社が本サービスの利用に応じて会員にポイントを付与すること」および「ポイントの有効期限が付与日から1年間であること」に関する規定です。


In [6]:
# 条ごとの要点から、改定全体の要約を作る
items = "\n".join(f"- {r['区分']} ({r['見出し']}): {r['要点']}" for _, r in changed.iterrows())
print(chat(f"利用規約の改定内容を条項ごとにまとめたものです。会員への影響が大きい順に、5 項目以内の箇条書きにしてください。各項目は 1 文にしてください。\n\n{items}", max_new_tokens=400))

利用規約の改定内容の中で、会員への影響が大きい順に以下の5つの項目を挙げます。

1. **料金及び支払方法の変更**: 月額料金が50円から増額され、月額500円から550円（税込）に変更されました。これにより、会員にとっては不利となり、負担が増えました。
2. **禁止事項の強化**: 本サービスの利用にあたり、特にデータの利用に関する制限が強化されました。具体的には、当社が許可しない形で本サービスで提供される情報を機械学習に利用することが禁止されるようになりました。これにより、会員のデータ利用に関する自由度が減少し、不利になります。
3. **パスワード管理の責任強化**: 会員はパスワードを自己管理する責任があり、さらにパスワードが第三者に知られたおそれがある場合に通知する義務が追加されました。これにより、会員にとっては通知義務が生じ、管理の負担が増えました。
4. **責任範囲の縮小**: 当社が本サービスに関して会員に生じた損害に対して、重大な過失がない限り責任を負わないという旧規定から、故意がない限り責任を負わないという変更がなされました。これにより、会員にとっては責任の範囲が狭まり、不利になる可能性があります。
5. **削除されたポイント制度**: ポイント制度に関する条項が削除されました。具体的には、「当社が本サービスの利用に応じて会員にポイントを付与すること」および「ポイントの有効期限が付与日から1年間であること」に関する規定が削除されたため、会員にとっては直接的な影響は少ないかもしれませんが、ポイントによる特典を利用していた場合には間接的に影響があります。ただし、影響の大きさとしては他の項目ほどではありません。

新設された「定義」に関する条項は、会員と投稿データを明確に定義するもので、規約の解釈を明確にするものであり、直接的な会員への影響は少ないと考えられます。


- 条番号だけで対応づけると、条項を 1 つ新設しただけで、新設と削除の間にある条（ここでは第2〜9条）がすべて「変更」に見える（最初の表。第10条は削除で番号のずれが打ち消されて一致した）。見出しと本文の類似度で対応づけ、`scipy.optimize.linear_sum_assignment` で割当問題を解いて 1 対 1 の組を選ぶと、番号のずれ、新設、削除を区別できる。
- 類似度は、文埋め込みと文字列の重なり (difflib) の平均にした。multilingual-e5-small のコサイン類似度は無関係な条どうしでも高く、上の出力では対応した組の最小 0.98 に対しそれ以外の最大が 0.94 だった。平均を取ると 0.90 と 0.66 に開くので、しきい値（ここでは 0.75）を置きやすい。見出しや言い回しを大きく書き換えた条では値が下がるので、しきい値は表を見て決める。
- 本文中の「第4条に違反した場合」のような条番号の参照は、番号が繰り下がっただけでも差分になる。対応づけでは伏せ、比べるときは旧版の番号を新版の番号に読み替えると、実質的な変更だけが残る（上の退会の条は「変更なし」になった）。
- 「することができます」→「しなければなりません」は文全体の類似度にほとんど表れない（パスワードの条を含め、対応した組の文埋め込みはすべて 0.98 以上）が、任意の通知が義務に変わる重要な改定である。変更の有無は文字列の一致で判定し、義務・禁止・免責などの語句と数値の数の変化を「要注意の語句」の列に出しておく。ただし、号の追加のように語句の数が変わらない変更はこの列に表れない（上の禁止事項の条）。差分の全文も確かめる。
- LLM には差分と語句の変化も渡し、どこが変わったかを先に示しておく。それでも細部は落ちる。上の出力では、免責の条の要点が旧版を「重大な過失がない限り」と説明して「故意又は」を落とし、全体の要約にもその誤りが引き継がれた。全体の要約には「月額料金が50円から増額され」という崩れた表現も混ざり、「各項目は 1 文」の指示も守られていない。有利・不利の判断もモデルの意見にすぎないので、最終的な判断は差分の一覧で行う。
- 実際の規約では、PDF や HTML から条の区切りを正しく取り出すのに手間がかかることが多い（このレシピでは試していない）。項や号（「2」「(1)」）の単位まで分けると、どこが変わったかをさらに細かく示せる。
- データとモデルの出典とライセンス: 新旧の規約は説明のために作った架空のもの。intfloat/multilingual-e5-small (MIT)、sbintuitions/sarashina2.2-3b-instruct-v0.1（SB Intuitions、MIT）。sarashina のトークナイザーを直すコードについては、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。